# RAG Playground (Clean)

Erstes Test-Notebook fuer Retrieval und optionale RAG-Latenz.
- nutzt bestehenden Chroma-Index (oder baut ihn optional neu)
- misst BM25 / dense / hybrid Retrieval-Zeiten
- zeigt Top-Chunks zur Query
- optional End-to-End mit Ollama


In [2]:
# Setup: Projekt-Root (ohne autoreload)
import os
import sys
from pathlib import Path

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if ROOT.exists():
    os.chdir(ROOT)

print("cwd:", cwd)
print("ROOT:", ROOT)
print("Working dir:", Path.cwd())
print("Python:", sys.executable)


cwd: c:\Users\maumo\OneDrive\Dokumente\rag_project
ROOT: c:\Users\maumo\OneDrive\Dokumente\rag_project
Working dir: c:\Users\maumo\OneDrive\Dokumente\rag_project
Python: c:\Users\maumo\OneDrive\Dokumente\rag_project\.venv\Scripts\python.exe


In [5]:
# Imports + Projektstatus
from src.config import RAW_DIR, CHROMA_DIR, DEFAULT_LLM_MODEL
from src.indexing import build_index, load_index, clear_retrieval_caches
from src.rag import get_docs_for_query, answer_with_rag_mode

pdfs = sorted(RAW_DIR.rglob('*.pdf')) if RAW_DIR.exists() else []
print('RAW_DIR:', RAW_DIR)
print('CHROMA_DIR:', CHROMA_DIR)
print('PDF count:', len(pdfs))
for p in pdfs[:10]:
    print('-', p.name)
if len(pdfs) > 10:
    print(f'... ({len(pdfs)-10} more)')
print('Index dir exists:', CHROMA_DIR.exists())
print('Default LLM model:', DEFAULT_LLM_MODEL)


ModuleNotFoundError: No module named 'src'

In [4]:
# Index laden (oder bei Bedarf neu bauen)
REBUILD_INDEX = True  # auf True setzen, wenn du Chunks/Index neu erstellen willst
CHUNK_SIZE = 1200
CHUNK_OVERLAP = 200

if REBUILD_INDEX:
    print('Rebuilding index ...')
    vs = build_index(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    print('Index rebuilt and persisted.')
else:
    vs = load_index()
    print('Existing index loaded.')

# Optional: nach Daten-/Code-Aenderungen Caches resetten
# clear_retrieval_caches()


Rebuilding index ...


c:\Users\maumo\OneDrive\Dokumente\rag_project\src\indexing.py:24: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceBgeEmbeddings(
Loading weights: 100%|██████████| 391/391 [00:01<00:00, 274.93it/s, Materializing param=pooler.dense.weight]                               


KeyboardInterrupt: 

In [3]:
# Helpers: Retrieval-Benchmark + Ausgabe der Top-Chunks
import time
from statistics import mean

def _preview(text: str, n: int = 220) -> str:
    text = ' '.join((text or '').split())
    return text[:n] + ('...' if len(text) > n else '')

def show_docs(docs, title='Docs'):
    print(f'\n=== {title} | count={len(docs)} ===')
    for i, d in enumerate(docs, start=1):
        md = d.metadata or {}
        print(f'{i}. source={md.get("source")}, page={md.get("page")}, section={md.get("section", "?")}')
        print('   ' + _preview(d.page_content))

def benchmark_retrieval(query: str, modes=('bm25', 'dense', 'hybrid'), k: int = 5, runs: int = 3, chunk_size: int = 1200, chunk_overlap: int = 200):
    print(f'Query: {query}')
    print(f'Modes={modes}, k={k}, runs={runs}, chunk_size={chunk_size}, chunk_overlap={chunk_overlap}')
    print('-' * 90)
    results = {}

    for mode in modes:
        latencies = []
        last_docs = []
        for idx in range(runs):
            t0 = time.perf_counter()
            docs = get_docs_for_query(
                query=query,
                mode=mode,
                k=k,
                chunk_size=chunk_size,
                chunk_overlap=chunk_overlap,
            )
            dt = time.perf_counter() - t0
            latencies.append(dt)
            last_docs = docs
            print(f'[{mode}] run {idx+1}/{runs}: {dt:.3f}s | docs={len(docs)}')

        avg_s = mean(latencies) if latencies else float('nan')
        results[mode] = {'latencies': latencies, 'avg_s': avg_s, 'docs': last_docs}
        print(f'[{mode}] avg: {avg_s:.3f}s')
        print('-' * 90)

    print('\nTop-Chunks aus letztem Lauf je Modus:')
    for mode in modes:
        show_docs(results[mode]['docs'], title=mode.upper())

    return results


In [ ]:
# Fragen zentral laden (id + query)
import json
from pathlib import Path

QUESTIONS_PATH = ROOT / 'data' / 'eval' / 'questions.jsonl'

def load_questions(path=QUESTIONS_PATH):
    items = {}
    with Path(path).open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            items[obj['id']] = obj['query']
    return items

questions = load_questions()
print('Questions file:', QUESTIONS_PATH)
print('Verfuegbare IDs:', ', '.join(sorted(questions.keys())))

qid = 'q1'
query = questions[qid]
print('Ausgewaehlte ID:', qid)
print('Frage:', query)


In [4]:
# Erster Retrieval-Test (ohne LLM)
# query kommt aus der Frage-ID-Zelle oben
retrieval_results = benchmark_retrieval(
    query=query,
    modes=('bm25', 'dense', 'hybrid'),
    k=5,
    runs=3,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)


NameError: name 'CHUNK_SIZE' is not defined

In [2]:
# Optional: End-to-End RAG-Latenz (nur wenn Ollama laeuft)
RUN_RAG = True
RAG_MODE = 'hybrid'  # 'bm25' | 'dense' | 'hybrid'

if RUN_RAG:
    t0 = time.perf_counter()
    rag_res = answer_with_rag_mode(
        query=query,
        mode=RAG_MODE,
        k=5,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )
    dt = time.perf_counter() - t0
    print(f'RAG mode={RAG_MODE} total={dt:.3f}s')
    print('\nAntwort:\n', rag_res['answer'])
    print('\nQuellen:')
    for s in rag_res.get('sources', []):
        print(f"- {s.get('source')} (Seite {s.get('page')})")
else:
    print('RUN_RAG=False -> nur Retrieval-Benchmark ausgefuehrt.')


NameError: name 'time' is not defined